# Judge validation — does the LLM correctness judge agree with a human?

**The most important remaining item.** Every correctness figure in the project comes from an LLM deciding whether an answer matches the gold. Nobody has checked that judge against a person — so the whole generation half currently rests on an unvalidated instrument. This is the first thing a methods reviewer will question.

**Two passes, with your labelling in between:**

1. Run cells 1–4 → produces `labeling_sheet.csv`, downloads it
2. Open in Excel/Sheets, fill `human_correct` with 1 or 0 for each row (~15–20 min for 100 short factoid answers)
3. Re-upload the filled sheet, run cells 5–7 → Cohen's κ

**One methodological note:** don't look at the `llm_correct` column while labelling. Seeing the judge's answer first biases your label and inflates the agreement score.

**Upload:** a generation CSV (or both, to validate across runs) + `qa_pairs_wiki.json`. **No GPU needed.**

### Install

In [ ]:
# !pip install -q pandas scikit-learn

### Config

In [ ]:
CONFIG = {
    # Any generation results CSV produced by the generation notebooks.
    # To validate BOTH rounds at once, list both files here.
    "generation_csvs": [
        "generation_reranked_raw.csv",
        "generation_n200_raw.csv",
    ],
    "qa_file": "qa_pairs_wiki.json",
    "n_per_file": 50,        # sampled per CSV; 50 x 2 = 100 rows to label
    "sheet_out": "labeling_sheet.csv",
    "seed": 42,
}

### Build the labelling sheet

In [ ]:
import json, os
import pandas as pd

with open(CONFIG["qa_file"], encoding="utf-8") as f:
    qa_lookup = {q["id"]: q for q in json.load(f)}

def stratified_sample(df, n, run_label):
    """Sample across conditions AND deliberately oversample the cases the judge
    called WRONG. A purely random sample would be dominated by obviously-correct
    answers, where agreement is trivial and uninformative; disagreements
    concentrate in the harder cases."""
    n_per = max(1, n // max(df["condition"].nunique(), 1))
    parts = []
    for cond, grp in df.groupby("condition"):
        wrong = grp[grp["correct"] == 0]
        right = grp[grp["correct"] == 1]
        n_wrong = min(len(wrong), max(1, n_per // 2))
        n_right = min(len(right), n_per - n_wrong)
        parts.append(pd.concat([
            wrong.sample(n_wrong, random_state=CONFIG["seed"]) if n_wrong else wrong,
            right.sample(n_right, random_state=CONFIG["seed"]) if n_right else right,
        ]))
    sample = (pd.concat(parts)
              .sample(frac=1, random_state=CONFIG["seed"])
              .reset_index(drop=True)
              .head(n))
    rows = []
    for _, r in sample.iterrows():
        q = qa_lookup.get(r["qid"], {})
        rows.append({
            "run": run_label, "qid": r["qid"], "condition": r["condition"],
            "msa_query": q.get("msa_query", ""),
            "darija_query": q.get("darija_query", ""),
            "gold_answer": q.get("gold_answer", ""),
            "model_answer": r.get("answer", ""),
            "llm_correct": int(r["correct"]),
            "human_correct": "",     # <-- you fill this in
        })
    return pd.DataFrame(rows)

sheets = []
for path in CONFIG["generation_csvs"]:
    if not os.path.exists(path):
        print(f"  '{path}' not found -- skipping.")
        continue
    df = pd.read_csv(path)
    if "correct" not in df.columns:
        print(f"  '{path}' has no 'correct' column -- skipping.")
        continue
    label = os.path.splitext(os.path.basename(path))[0]
    sheets.append(stratified_sample(df, CONFIG["n_per_file"], label))
    print(f"  sampled {CONFIG['n_per_file']} from {path}")

if not sheets:
    raise RuntimeError("No usable generation CSVs found -- upload at least one.")

combined = pd.concat(sheets, ignore_index=True)
# utf-8-sig so Excel displays the Arabic text correctly
combined.to_csv(CONFIG["sheet_out"], index=False, encoding="utf-8-sig")
print(f"\nWrote {len(combined)} rows to {CONFIG['sheet_out']}")
print(combined.groupby("run").size().to_string())
print(f"\nLLM said correct: {combined['llm_correct'].sum()}/{len(combined)}")

### Download the sheet, then label it

In [ ]:
from google.colab import files
files.download(CONFIG["sheet_out"])

print("""
NEXT STEPS
  1. Open the downloaded labeling_sheet.csv in Excel or Google Sheets.
  2. For each row, read 'gold_answer' against 'model_answer'.
  3. Put 1 in 'human_correct' if the model's answer is correct, 0 if not.
     Wording may differ freely; numbers, names and dates must match.
     Do NOT look at the 'llm_correct' column while labelling -- seeing the
     judge's answer first biases the human label and inflates agreement.
  4. Save as CSV, re-upload it here, then run the remaining cells.
""")

### Load the filled sheet

In [ ]:
import numpy as np

SHEET = "labeling_sheet.csv"   # re-upload the filled version under this name
df = pd.read_csv(SHEET)

raw = df["human_correct"]
unfilled = raw.isna() | raw.astype(str).str.strip().eq("")
if unfilled.any():
    raise ValueError(f"{unfilled.sum()} row(s) are unlabelled "
                     f"(qid: {df.loc[unfilled, 'qid'].tolist()[:10]}...). Fill them all first.")

human = raw.astype(int)
llm = df["llm_correct"].astype(int)
if not human.isin([0, 1]).all():
    raise ValueError("'human_correct' must contain only 0 or 1.")
print(f"Loaded {len(df)} labelled rows.")

### Cohen's kappa

In [ ]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix, accuracy_score

kappa = cohen_kappa_score(human, llm)
acc = accuracy_score(human, llm)
cm = confusion_matrix(human, llm, labels=[0, 1])

print("=" * 62)
print("JUDGE VALIDATION")
print("=" * 62)
print(f"n = {len(df)}")
print(f"Raw agreement:  {acc:.3f}")
print(f"Cohen's kappa:  {kappa:.3f}")
print()
print("Confusion matrix (rows = human, cols = LLM judge):")
print("              LLM=0    LLM=1")
print(f"  human=0     {cm[0,0]:>5}    {cm[0,1]:>5}")
print(f"  human=1     {cm[1,0]:>5}    {cm[1,1]:>5}")

# Landis & Koch (1977) interpretation bands
if kappa < 0.20:   band = "slight — the judge is not usable as reported"
elif kappa < 0.40: band = "fair — weak; revise the judge prompt"
elif kappa < 0.60: band = "moderate — borderline; report with explicit caution"
elif kappa < 0.80: band = "substantial — defensible for reporting"
else:              band = "almost perfect — strong"
print(f"\nInterpretation: {band}")

# Kappa is deflated when one class dominates, so report the balance too.
print(f"\nClass balance (human labels): {human.mean():.1%} correct")
if human.mean() > 0.85 or human.mean() < 0.15:
    print("  Note: with a heavily skewed class balance, kappa is pessimistic.")
    print("  Report raw agreement alongside it.")

### Inspect disagreements, and per-run breakdown

In [ ]:
dis = df[human != llm]
print("\n" + "=" * 62)
print(f"DISAGREEMENTS ({len(dis)} of {len(df)})")
print("=" * 62)
if len(dis):
    for _, r in dis.iterrows():
        print(f"\n[{r['run']} | {r['condition']}]")
        print(f"  gold:  {r['gold_answer']}")
        print(f"  model: {str(r['model_answer'])[:200]}")
        print(f"  llm={r['llm_correct']}  human={r['human_correct']}")
else:
    print("None — perfect agreement.")

if df["run"].nunique() > 1:
    print("\n" + "=" * 62)
    print("PER-RUN AGREEMENT")
    print("=" * 62)
    for run, grp in df.groupby("run"):
        h = grp["human_correct"].astype(int)
        l = grp["llm_correct"].astype(int)
        print(f"  {run:<32} n={len(grp):<4} kappa={cohen_kappa_score(h, l):.3f}  "
              f"agreement={accuracy_score(h, l):.3f}")
    print("\n  Similar kappa across runs means the judge behaves consistently")
    print("  regardless of the retrieval setup being evaluated.")

df.to_csv("labeling_sheet_scored.csv", index=False, encoding="utf-8-sig")
print(f"""
FOR THE PAPER
  "Correctness labels were assigned by an LLM judge. On a stratified sample of
   {len(df)} generated answers spanning [conditions/runs], LLM and human labels
   agreed with Cohen's kappa = {kappa:.2f} (raw agreement {acc:.1%}), indicating
   {band.split(' — ')[0]} agreement."
""")
files.download("labeling_sheet_scored.csv")